In [2]:
import pandas as pd

input_path = "kpi_with_anomalies.csv"
output_path = "data_2026.csv"

df = pd.read_csv(input_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])
def shift_year(t):
    new_year = t.year + 10
    if t.month == 2 and t.day == 29:
        return t.replace(year=new_year, day=28)
    return t.replace(year=new_year)

df["timestamp"] = df["timestamp"].apply(shift_year)
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("时间戳范围:")
print(f"  最早: {df['timestamp'].min()}")
print(f"  最晚: {df['timestamp'].max()}")
print(f"共 {len(df)} 行已写入 {output_path}")

时间戳范围:
  最早: 2026-01-01 00:00:00
  最晚: 2026-12-31 23:00:00
共 26316 行已写入 data_2026.csv


In [3]:
cols = ["timestamp", "building_id", "primaryspaceusage", "sqm",
        "airTemperature", "dewTemperature", "humidity_rh_pct"]

df_sub = pd.read_csv("data_2026.csv", usecols=cols)
df_sub = df_sub[cols]
df_sub.to_csv("data_2026_selected.csv", index=False, encoding="utf-8-sig")

print(f"提取 {len(cols)} 列，共 {len(df_sub)} 行 → data_2026_selected.csv")
df_sub.head()

提取 7 列，共 26316 行 → data_2026_selected.csv


,timestamp,building_id,primaryspaceusage,sqm,airTemperature,dewTemperature,humidity_rh_pct
0,2026-01-01 00:00:00,Fox_education_Louie,Education,1664.52,7.2,-5.6,37.96
1,2026-01-01 01:00:00,Fox_education_Louie,Education,1664.52,7.2,-6.7,34.84
2,2026-01-01 02:00:00,Fox_education_Louie,Education,1664.52,6.1,-5.0,44.64
3,2026-01-01 03:00:00,Fox_education_Louie,Education,1664.52,5.0,-5.6,44.54
4,2026-01-01 04:00:00,Fox_education_Louie,Education,1664.52,4.4,-5.6,46.18


In [12]:
import copy
import numpy as np
import pandas as pd

# ======== 输入/输出路径 ========
input_path = "data_2026_selected.csv"
output_path = "data_2026_simulated.csv"

df = pd.read_csv(input_path)

# ======== 1) 预处理：列名适配 & 派生 hour/dayofweek ========
for c in ["sqm", "airTemperature", "dewTemperature", "humidity_rh_pct"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

ts = pd.to_datetime(df["timestamp"], errors="coerce")
hour = ts.dt.hour.fillna(0).astype(int).clip(0, 23)
dow = (ts.dt.dayofweek + 1).fillna(1).astype(int).clip(1, 7)  # 1=Mon … 7=Sun

if "building_id" in df.columns:
    _n_dup = len(df)
    df = df.drop_duplicates(subset=["building_id", "timestamp"], keep="first").reset_index(drop=True)
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    hour = ts.dt.hour.fillna(0).astype(int).clip(0, 23)
    dow = (ts.dt.dayofweek + 1).fillna(1).astype(int).clip(1, 7)
    if len(df) < _n_dup:
        print(f"[dedupe] 去重删除 {_n_dup - len(df)} 行")

# ======== 2) 建筑分类 ========
BUILDING_ID_TO_TYPE = {"Fox_public_Henry": "library"}
DEFAULT_BUILDING_TYPE = "teaching"
PRIMARYSPACEUSAGE_TO_TYPE = {
    "office": "office", "education": "teaching",
    "public services": "library", "public service": "library", "public_services": "library",
}

def _norm_text(val):
    if val is None: return ""
    try:
        if isinstance(val, float) and np.isnan(val): return ""
    except TypeError: pass
    return str(val).strip().lower()

def classify_usage(row):
    bid = str(row.get("building_id", "")).strip()
    if bid in BUILDING_ID_TO_TYPE:
        return BUILDING_ID_TO_TYPE[bid]
    bt = row.get("building_type", None)
    if bt is not None and (not pd.isna(bt)) and str(bt).strip() != "":
        bt_s = str(bt).strip().lower()
        if bt_s in ("office","办公","办公楼","写字楼","行政"): return "office"
        if bt_s in ("teaching","教学","教学楼","教室","教育"): return "teaching"
        if bt_s in ("library","图书","图书馆","阅览"): return "library"
    psu_key = str(row.get("primaryspaceusage", "")).strip().lower()
    if psu_key in PRIMARYSPACEUSAGE_TO_TYPE:
        return PRIMARYSPACEUSAGE_TO_TYPE[psu_key]
    bid_l = bid.lower()
    if any(k in bid for k in ("办公","写字楼","行政")) or any(k in bid_l for k in ("office","admin","hq")): return "office"
    if any(k in bid for k in ("教学","教室")) or any(k in bid_l for k in ("teach","classroom","class_","edu")): return "teaching"
    if any(k in bid for k in ("图书","阅览")) or "library" in bid_l: return "library"
    psu = _norm_text(row.get("primaryspaceusage", ""))
    if "office" in psu or "办公" in psu: return "office"
    if any(k in psu for k in ("teaching","class","education","教室","教学")): return "teaching"
    if any(k in psu for k in ("library","图书","阅览")): return "library"
    return DEFAULT_BUILDING_TYPE

usage = df.apply(classify_usage, axis=1)
weekday = dow <= 5

# ======== 3) 时间系数 S(hour, dayofweek) ========
def S_office(h, is_wd):
    if not is_wd: return 0.08
    if 0<=h<=6: return 0.08
    if h==7: return 0.2
    if h==8: return 0.6
    if 9<=h<=17: return 1.0
    if 18<=h<=20: return 0.35
    return 0.10

def S_teaching(h, is_wd):
    if not is_wd: return 0.15
    if 0<=h<=6: return 0.08
    if h==7: return 0.2
    if 8<=h<=12: return 1.0
    if h==13: return 0.45
    if 14<=h<=17: return 1.0
    if 18<=h<=21: return 0.75
    return 0.10

def S_library(h, is_wd):
    if 0<=h<=6: return 0.10
    if h==7: return 0.2
    if 8<=h<=21: return 0.85
    return 0.15

S = np.zeros(len(df), dtype=float)
for i, (u, h, wd) in enumerate(zip(usage.values, hour.values, weekday.values)):
    if u == "office": S[i] = S_office(h, wd)
    elif u == "teaching": S[i] = S_teaching(h, wd)
    else: S[i] = S_library(h, wd)
S = np.clip(S, 0.08, 1.0)

# ======== 4) 国标参数模板 ========
GB_PARAM_TEMPLATE = {
    "office": {
        "internal": {"W_peak":0.36,"P_base":22.0,"q_person":12.0,"U_envelope":5.0,"water_base_wm2":0.004,"water_weather_wm2_per_deg":0.002},
        "hvac": {"cop":{"cooling":3.6,"heating":2.7},"pump_wm2":{"run":1.00,"standby":0.15},"vent_wm2":{"active":2.2,"minimum":0.6},
                 "hysteresis":{"cool_on_ta":24.0,"cool_off_ta":22.0,"cool_on_q":22.0,"cool_off_q":14.0,"heat_on_ta":14.0,"heat_off_ta":16.0,"heat_on_qenv":-18.0},
                 "extreme_force_on":{"low_ta":8.0,"high_ta":30.0},"min_on_load_kw":0.05},
    },
    "teaching": {
        "internal": {"W_peak":0.48,"P_base":24.0,"q_person":13.0,"U_envelope":5.3,"water_base_wm2":0.008,"water_weather_wm2_per_deg":0.0022},
        "hvac": {"cop":{"cooling":3.4,"heating":2.6},"pump_wm2":{"run":1.05,"standby":0.16},"vent_wm2":{"active":2.6,"minimum":0.7},
                 "hysteresis":{"cool_on_ta":24.0,"cool_off_ta":22.0,"cool_on_q":24.0,"cool_off_q":15.0,"heat_on_ta":14.0,"heat_off_ta":16.0,"heat_on_qenv":-20.0},
                 "extreme_force_on":{"low_ta":7.0,"high_ta":31.0},"min_on_load_kw":0.05},
    },
    "library": {
        "internal": {"W_peak":0.30,"P_base":20.0,"q_person":11.0,"U_envelope":4.6,"water_base_wm2":0.0028,"water_weather_wm2_per_deg":0.0015},
        "hvac": {"cop":{"cooling":3.8,"heating":2.9},"pump_wm2":{"run":0.85,"standby":0.12},"vent_wm2":{"active":1.8,"minimum":0.5},
                 "hysteresis":{"cool_on_ta":24.5,"cool_off_ta":22.5,"cool_on_q":20.0,"cool_off_q":13.0,"heat_on_ta":13.5,"heat_off_ta":16.0,"heat_on_qenv":-16.0},
                 "extreme_force_on":{"low_ta":7.0,"high_ta":31.0},"min_on_load_kw":0.04},
    },
}

UNIFY_CLIMATE_AND_PLANT_ACROSS_TYPES = True
if UNIFY_CLIMATE_AND_PLANT_ACROSS_TYPES:
    ref = GB_PARAM_TEMPLATE["office"]
    ref_hvac = copy.deepcopy(ref["hvac"])
    ref_u = ref["internal"]["U_envelope"]
    ref_wb = ref["internal"]["water_base_wm2"]
    ref_ww = ref["internal"]["water_weather_wm2_per_deg"]
    for u in GB_PARAM_TEMPLATE:
        GB_PARAM_TEMPLATE[u]["internal"]["U_envelope"] = ref_u
        GB_PARAM_TEMPLATE[u]["internal"]["water_base_wm2"] = ref_wb
        GB_PARAM_TEMPLATE[u]["internal"]["water_weather_wm2_per_deg"] = ref_ww
        GB_PARAM_TEMPLATE[u]["hvac"] = copy.deepcopy(ref_hvac)

def param_array(usage_s, ext):
    return usage_s.map(lambda u: ext(GB_PARAM_TEMPLATE[u])).to_numpy()

W_peak = param_array(usage, lambda x: x["internal"]["W_peak"])
P_base = param_array(usage, lambda x: x["internal"]["P_base"])
q_person = param_array(usage, lambda x: x["internal"]["q_person"])
cop_cooling = param_array(usage, lambda x: x["hvac"]["cop"]["cooling"])
cop_heating = param_array(usage, lambda x: x["hvac"]["cop"]["heating"])
pump_run_wm2 = param_array(usage, lambda x: x["hvac"]["pump_wm2"]["run"])
pump_standby_wm2 = param_array(usage, lambda x: x["hvac"]["pump_wm2"]["standby"])
vent_active_wm2 = param_array(usage, lambda x: x["hvac"]["vent_wm2"]["active"])
vent_min_wm2 = param_array(usage, lambda x: x["hvac"]["vent_wm2"]["minimum"])
cool_on_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_on_ta"])
cool_off_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_off_ta"])
cool_on_q = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_on_q"])
cool_off_q = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_off_q"])
heat_on_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["heat_on_ta"])
heat_off_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["heat_off_ta"])
heat_on_qenv = param_array(usage, lambda x: x["hvac"]["hysteresis"]["heat_on_qenv"])
force_on_low_ta = param_array(usage, lambda x: x["hvac"]["extreme_force_on"]["low_ta"])
force_on_high_ta = param_array(usage, lambda x: x["hvac"]["extreme_force_on"]["high_ta"])
min_on_load_kw = param_array(usage, lambda x: x["hvac"]["min_on_load_kw"])
U_env = param_array(usage, lambda x: x["internal"]["U_envelope"])
water_base_wm2 = param_array(usage, lambda x: x["internal"]["water_base_wm2"])
water_weather_wm2 = param_array(usage, lambda x: x["internal"]["water_weather_wm2_per_deg"])

sqm = df["sqm"].fillna(df["sqm"].median()).to_numpy()
Ta = df["airTemperature"].ffill().bfill().fillna(20.0).to_numpy()
Td = df["dewTemperature"].ffill().bfill().fillna(10.0).to_numpy()
RH_base = df["humidity_rh_pct"].ffill().bfill().fillna(50.0).to_numpy()

# ======== 5) 季节判定 ========
month = ts.dt.month.fillna(0).astype(int).to_numpy()
season = np.full(len(df), "transition", dtype=object)
season[np.isin(month, [6,7,8,9])] = "cooling"
season[np.isin(month, [11,12,1,2,3])] = "heating"

# ======== 6) 用水仿真 (L) ========
rng = np.random.default_rng(20260320)
water_active = sqm * W_peak * S
water_base = sqm * water_base_wm2
water_weather = np.maximum(Ta - 26.0, 0.0) * sqm * water_weather_wm2
water_ideal = water_active + water_base + water_weather
water_noise = rng.normal(0.0, 0.06 * np.maximum(water_ideal, 1e-6), size=len(df))
water_l = np.maximum(0.0, water_ideal + water_noise)
water_l_nominal = np.maximum(0.0, water_ideal)

# ======== 7) Q_load & 工况滞回判定 ========
T_in = np.where(Ta < 18.0, 20.0, 26.0)
q_env = U_env * (Ta - T_in)
q_internal = (P_base * S) + (q_person * S)
q_latent = np.maximum(0.0, 15.0 * (Td - 15.0)) * S
q_total = q_env + q_internal + q_latent

cool_demand = (Ta >= cool_on_ta) | (q_total >= cool_on_q)
heat_demand = (Ta <= heat_on_ta) | ((q_env <= heat_on_qenv) & (Ta < (heat_off_ta + 1.0)))

is_cooling = np.zeros(len(df), dtype=bool)
is_heating = np.zeros(len(df), dtype=bool)

group_key = df["building_id"].fillna("unknown").astype(str) if "building_id" in df.columns else pd.Series(["all"]*len(df), index=df.index)
ts_sort = pd.to_datetime(df["timestamp"], errors="coerce")

for bid in group_key.unique():
    idx = np.where(group_key.values == bid)[0]
    ord_idx = idx[np.argsort(ts_sort.iloc[idx].values)]
    mode = 0
    for j in ord_idx:
        if mode == 1 and (Ta[j] <= cool_off_ta[j]) and (q_total[j] <= cool_off_q[j]): mode = 0
        elif mode == -1 and (Ta[j] >= heat_off_ta[j]): mode = 0
        if mode == 0:
            if heat_demand[j]: mode = -1
            elif cool_demand[j]: mode = 1
        is_cooling[j] = (mode == 1)
        is_heating[j] = (mode == -1)

Q_load = np.zeros(len(df), dtype=float)
Q_load[is_cooling] = sqm[is_cooling] * np.maximum(0.0, q_env[is_cooling]+q_internal[is_cooling]+q_latent[is_cooling]) / 1000.0
Q_load[is_heating] = sqm[is_heating] * np.maximum(0.0, np.abs(q_env[is_heating])-q_internal[is_heating]) / 1000.0

hvac_on = (Q_load > min_on_load_kw) & ((S > 0.08) | (Ta < force_on_low_ta) | (Ta > force_on_high_ta))
Q_load = np.where(hvac_on, Q_load, 0.0)
Q_design = sqm * 100.0 / 1000.0
PLR = np.clip(np.divide(Q_load, np.maximum(Q_design, 1e-6)), 0.0, 1.0)

# ======== 8) 供回水温度 ========
sim_supply_nominal = Ta.astype(float).copy()
sim_return_nominal = Ta.astype(float).copy()
idx_c = is_cooling & hvac_on; idx_h = is_heating & hvac_on
sim_supply_nominal[idx_c] = 7.0;  sim_return_nominal[idx_c] = 7.0 + 5.0*PLR[idx_c]
sim_supply_nominal[idx_h] = 60.0; sim_return_nominal[idx_h] = 60.0 - 10.0*PLR[idx_h]

sim_supply_temp_c = sim_supply_nominal.copy()
sim_return_temp_c = sim_return_nominal.copy()
sim_supply_temp_c[idx_c] += rng.normal(0,0.2,size=idx_c.sum())
sim_return_temp_c[idx_c] += rng.normal(0,0.2,size=idx_c.sum())
sim_supply_temp_c[idx_h] += rng.normal(0,0.5,size=idx_h.sum())
sim_return_temp_c[idx_h] += rng.normal(0,0.5,size=idx_h.sum())

# ======== 9) hac_energy (COP + 泵 + 新风) ========
COP_EFFECTIVE_STOCHASTIC_ENABLED = True
n_h = len(df)
E_pump_run = sqm * pump_run_wm2 / 1000.0
E_pump_standby = sqm * pump_standby_wm2 / 1000.0
E_vent_active = np.where(S > 0.12, sqm*vent_active_wm2/1000.0, sqm*vent_min_wm2/1000.0)

if COP_EFFECTIVE_STOCHASTIC_ENABLED:
    mult_c = np.exp(rng.normal(-0.5*0.045**2, 0.045, size=n_h))
    mult_h = np.exp(rng.normal(-0.5*0.055**2, 0.055, size=n_h))
    cop_eff_c = np.clip(cop_cooling*mult_c, 2.3, 6.0)
    cop_eff_h = np.clip(cop_heating*mult_h, 1.9, 4.2)
else:
    cop_eff_c = cop_cooling.astype(float).copy()
    cop_eff_h = cop_heating.astype(float).copy()

cool_idx = is_cooling & hvac_on; heat_idx = is_heating & hvac_on; tran_idx = ~(cool_idx|heat_idx)

hac_energy = np.zeros(n_h, dtype=float)
hac_energy[cool_idx] = Q_load[cool_idx]/np.maximum(cop_eff_c[cool_idx],1e-6) + E_pump_run[cool_idx] + 0.5*E_vent_active[cool_idx]
hac_energy[heat_idx] = Q_load[heat_idx]/np.maximum(cop_eff_h[heat_idx],1e-6) + E_pump_run[heat_idx] + 0.4*E_vent_active[heat_idx]
hac_energy[tran_idx] = E_pump_standby[tran_idx] + E_vent_active[tran_idx]

hac_energy_nominal = np.zeros(n_h, dtype=float)
hac_energy_nominal[cool_idx] = Q_load[cool_idx]/np.maximum(cop_cooling[cool_idx],1e-6) + E_pump_run[cool_idx] + 0.5*E_vent_active[cool_idx]
hac_energy_nominal[heat_idx] = Q_load[heat_idx]/np.maximum(cop_heating[heat_idx],1e-6) + E_pump_run[heat_idx] + 0.4*E_vent_active[heat_idx]
hac_energy_nominal[tran_idx] = E_pump_standby[tran_idx] + E_vent_active[tran_idx]

# ======== 10) electricity_kwh ========
E_base = (sqm * P_base * S) / 1000.0
elec_noise = rng.normal(0.0, 0.04*np.maximum(E_base, 1e-6), size=n_h)
electricity_kwh_nominal = np.maximum(0.0, E_base + hac_energy_nominal)
electricity_kwh = np.maximum(0.0, E_base + hac_energy + elec_noise)

# ======== 11) humidity_rh_pct ========
humidity_rh_pct = RH_base.copy().astype(float)
idx = is_cooling & hvac_on
humidity_rh_pct[idx] -= (7.0+11.0*PLR[idx])*np.maximum(S[idx],0.12)
idx = is_heating & hvac_on
humidity_rh_pct[idx] -= (3.0+5.0*PLR[idx])*np.maximum(S[idx],0.12)
off_idx = ~hvac_on
humidity_rh_pct[off_idx] = 0.9*humidity_rh_pct[off_idx] + 0.1*RH_base[off_idx]
humidity_rh_pct_nominal = np.clip(humidity_rh_pct.copy(), 20.0, 95.0)
humidity_rh_pct += rng.normal(0.0, 1.2, size=n_h)
humidity_rh_pct = np.clip(humidity_rh_pct, 20.0, 95.0)

# ======== 12) 裁剪噪声尾部：确保偏差不超过软检测阈值 ========
SOFT_FACTOR = 0.62
CLIP_MARGIN = 0.90  # 留 10% 余量，确保不会刚好压线

MAX_SUPPLY_DEV = 1.5 * SOFT_FACTOR * CLIP_MARGIN   # 0.837°C
MAX_RETURN_DEV = 2.0 * SOFT_FACTOR * CLIP_MARGIN   # 1.116°C
MAX_WATER_REL  = 0.20 * SOFT_FACTOR * CLIP_MARGIN  # 11.2%
MAX_ELEC_REL   = 0.15 * SOFT_FACTOR * CLIP_MARGIN  # 8.4%
MAX_HVAC_REL   = 0.20 * SOFT_FACTOR * CLIP_MARGIN  # 11.2%

sim_supply_temp_c = np.clip(
    sim_supply_temp_c,
    sim_supply_nominal - MAX_SUPPLY_DEV,
    sim_supply_nominal + MAX_SUPPLY_DEV,
)
sim_return_temp_c = np.clip(
    sim_return_temp_c,
    sim_return_nominal - MAX_RETURN_DEV,
    sim_return_nominal + MAX_RETURN_DEV,
)
water_l = np.clip(
    water_l,
    water_l_nominal * (1 - MAX_WATER_REL),
    water_l_nominal * (1 + MAX_WATER_REL),
)
electricity_kwh = np.clip(
    electricity_kwh,
    electricity_kwh_nominal * (1 - MAX_ELEC_REL),
    electricity_kwh_nominal * (1 + MAX_ELEC_REL),
)
hac_energy = np.clip(
    hac_energy,
    hac_energy_nominal * (1 - MAX_HVAC_REL),
    hac_energy_nominal * (1 + MAX_HVAC_REL),
)
humidity_rh_pct = np.clip(
    humidity_rh_pct,
    humidity_rh_pct_nominal - 3.0,
    humidity_rh_pct_nominal + 3.0,
)
humidity_rh_pct = np.clip(humidity_rh_pct, 20.0, 95.0)

print(f"[clip] 噪声尾部已裁剪，各项最大偏差: "
      f"supply ±{MAX_SUPPLY_DEV:.3f}°C, return ±{MAX_RETURN_DEV:.3f}°C, "
      f"water ±{MAX_WATER_REL:.1%}, elec ±{MAX_ELEC_REL:.1%}, hvac ±{MAX_HVAC_REL:.1%}")

# ======== 13) 写回结果 ========
df["dayofweek"] = dow.values
df["water_l"] = water_l.round(2)
df["electricity_kwh"] = electricity_kwh.round(4)
df["sim_supply_temp_c"] = sim_supply_temp_c.round(3)
df["sim_return_temp_c"] = sim_return_temp_c.round(3)
df["hac_energy"] = hac_energy.round(4)
df["humidity_rh_pct"] = humidity_rh_pct.round(2)
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Done → {output_path}  ({len(df)} 行)")
print(df[["timestamp","building_id","primaryspaceusage",
          "water_l","electricity_kwh","sim_supply_temp_c","sim_return_temp_c",
          "hac_energy","humidity_rh_pct"]].head(10))
print("\n建筑类型分布:\n", usage.value_counts())

[dedupe] 去重删除 72 行
[clip] 噪声尾部已裁剪，各项最大偏差: supply ±0.837°C, return ±1.116°C, water ±11.2%, elec ±8.4%, hvac ±11.2%
Done → data_2026_simulated.csv  (26244 行)
             timestamp          building_id primaryspaceusage  water_l  \
0  2026-01-01 00:00:00  Fox_education_Louie         Education    66.49   
1  2026-01-01 01:00:00  Fox_education_Louie         Education    64.04   
2  2026-01-01 02:00:00  Fox_education_Louie         Education    74.18   
3  2026-01-01 03:00:00  Fox_education_Louie         Education    75.22   
4  2026-01-01 04:00:00  Fox_education_Louie         Education    64.51   
5  2026-01-01 05:00:00  Fox_education_Louie         Education    67.91   
6  2026-01-01 06:00:00  Fox_education_Louie         Education    70.04   
7  2026-01-01 07:00:00  Fox_education_Louie         Education   158.93   
8  2026-01-01 08:00:00  Fox_education_Louie         Education   770.95   
9  2026-01-01 09:00:00  Fox_education_Louie         Education   891.89   

   electricity_kwh  sim_suppl

In [13]:
import importlib
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "building_anomaly_detection.py").is_file():
    pass
elif (ROOT.parent / "building_anomaly_detection.py").is_file():
    ROOT = ROOT.parent
else:
    raise FileNotFoundError("未找到 building_anomaly_detection.py")

sys.path.insert(0, str(ROOT))
import building_anomaly_detection as _det
importlib.reload(_det)
from building_anomaly_detection import merge_expected_from_baseline, detect_building_kpi_anomalies

# ---- 用 Cell 2 内存中的 nominal 变量构造基线 DataFrame ----
baseline = pd.DataFrame({
    "timestamp": df["timestamp"],
    "building_id": df["building_id"],
    "water_l": water_l_nominal.round(2),
    "electricity_kwh": electricity_kwh_nominal.round(4),
    "hac_energy": hac_energy_nominal.round(4),
    "sim_supply_temp_c": sim_supply_nominal.round(3),
    "sim_return_temp_c": sim_return_nominal.round(3),
})
baseline["timestamp"] = pd.to_datetime(baseline["timestamp"])

# ---- 读取仿真结果 & 合并期望列 ----
df_det = pd.read_csv("data_2026_simulated.csv")
df_det["timestamp"] = pd.to_datetime(df_det["timestamp"])
df_det = df_det.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

df_det = merge_expected_from_baseline(df_det, baseline)
df_det = detect_building_kpi_anomalies(df_det, include_soft_symptoms=True)

# ---- 汇总结果 ----
print("=" * 60)
print("异常检测结果（数据无注入异常，理论上应全部为 0 或极少）")
print("=" * 60)
for flag in ["flag_water", "flag_electricity", "flag_hvac_energy",
             "flag_supply_deviation", "flag_return_deviation"]:
    print(f"  {flag:30s}: {int(df_det[flag].sum()):>5d}")
print(f"  {'flag_symptom_any':30s}: {int(df_det['flag_symptom_any'].sum()):>5d}")
print(f"  {'flag_symptom_soft_any':30s}: {int(df_det['flag_symptom_soft_any'].sum()):>5d}")
print(f"  {'flag_symptom_union_any':30s}: {int(df_det['flag_symptom_union_any'].sum()):>5d}")

n_total = len(df_det)
n_any = int(df_det["flag_symptom_union_any"].sum())
print(f"\n总行数: {n_total}，被标记行数: {n_any}，误报率: {n_any/n_total:.4%}")

# 如果有误报，展示几条看看
if n_any > 0:
    show_cols = ["timestamp", "building_id",
                 "flag_water", "flag_electricity", "flag_hvac_energy",
                 "flag_supply_deviation", "flag_return_deviation",
                 "hvac_diagnosis"]
    print("\n被标记的样例行:")
    display(df_det.loc[df_det["flag_symptom_union_any"], show_cols].head(10))
else:
    print("\n无任何异常标记，符合预期。")

异常检测结果（数据无注入异常，理论上应全部为 0 或极少）
  flag_water                    :     0
  flag_electricity              :     0
  flag_hvac_energy              :     0
  flag_supply_deviation         :     0
  flag_return_deviation         :     0
  flag_symptom_any              :     0
  flag_symptom_soft_any         :     0
  flag_symptom_union_any        :     0

总行数: 26244，被标记行数: 0，误报率: 0.0000%

无任何异常标记，符合预期。


In [16]:
import numpy as np
import pandas as pd

rng_inj = np.random.default_rng(20260329)

# ======== 1. 读取仿真数据 ========
df_inj = pd.read_csv("data_2026_simulated.csv")
df_inj["timestamp"] = pd.to_datetime(df_inj["timestamp"])
df_inj = df_inj.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

# ======== 2. 构建并保存基线（Cell 2 内存 nominal 值）========
_base_raw = pd.DataFrame({
    "timestamp": pd.to_datetime(df["timestamp"]),
    "building_id": df["building_id"].values,
    "water_l": water_l_nominal.round(2),
    "electricity_kwh": electricity_kwh_nominal.round(4),
    "hac_energy": hac_energy_nominal.round(4),
    "sim_supply_temp_c": sim_supply_nominal.round(3),
    "sim_return_temp_c": sim_return_nominal.round(3),
})
_base_save = _base_raw.copy()
_base_save["timestamp"] = _base_save["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
_base_save.to_csv("data_2026_baseline.csv", index=False, encoding="utf-8-sig")

_bm = _base_raw.rename(columns={
    c: f"{c}_base" for c in ["water_l","electricity_kwh","hac_energy",
                              "sim_supply_temp_c","sim_return_temp_c"]
})
df_inj = df_inj.merge(_bm, on=["timestamp","building_id"], how="left")

# ======== 3. 预计算索引数组 ========
ts_inj = df_inj["timestamp"]
hour_inj = ts_inj.dt.hour.to_numpy()
dow_inj  = ts_inj.dt.dayofweek.to_numpy()
month_inj = ts_inj.dt.month.to_numpy()

psu = df_inj["primaryspaceusage"].astype(str).str.strip().str.lower()
is_office   = psu.eq("office").values
is_teaching = psu.eq("education").values
is_library  = psu.isin(["public services","public service","library"]).values

used = np.zeros(len(df_inj), dtype=bool)
labels = []

def bv(i, col):
    return float(df_inj.at[i, f"{col}_base"])

# ======== 4. 13 种注入函数（幅度与 kpi_anomaly_inject.py 一致，仅 full 相位）========
def inject_1(blk):
    """办公楼周末排程失效：空调全力运行"""
    for i in blk:
        df_inj.at[i, "hac_energy"] = 85.0
        df_inj.at[i, "sim_supply_temp_c"] = 7.0
        df_inj.at[i, "sim_return_temp_c"] = 7.5

def inject_2(blk):
    """办公楼掩蔽效应：基载压低 + 空调拉高"""
    for i in blk:
        h0, e0 = bv(i,"hac_energy"), bv(i,"electricity_kwh")
        base0 = max(e0 - h0, 0.0)
        h1 = max(h0 * 1.25, h0 + 22.0)
        df_inj.at[i, "hac_energy"] = h1
        df_inj.at[i, "electricity_kwh"] = 0.7 * base0 + h1

def inject_3(blk):
    """办公楼大流量小温差旁通"""
    for i in blk:
        h0 = bv(i,"hac_energy")
        df_inj.at[i, "hac_energy"] = max(h0 * 1.35, h0 + 22.0)
        sup = float(df_inj.at[i,"sim_supply_temp_c"])
        df_inj.at[i, "sim_return_temp_c"] = sup + rng_inj.uniform(0.8, 1.45)

def inject_4(blk):
    """教学楼跑水"""
    mult = rng_inj.uniform(1.5, 2.0)
    for i in blk:
        df_inj.at[i, "water_l"] = float(df_inj.at[i,"water_l"]) * mult + 1100.0

def inject_5(blk):
    """教学楼开窗浪费制冷"""
    dret = rng_inj.uniform(3.0, 5.0)
    for i in blk:
        h0 = bv(i,"hac_energy")
        df_inj.at[i, "hac_energy"] = max(h0 * 1.4, h0 + 20.0)
        df_inj.at[i, "sim_return_temp_c"] = float(df_inj.at[i,"sim_return_temp_c"]) + dret

def inject_6(blk):
    """教学楼极端高温主机衰退"""
    sup_spike = float(rng_inj.uniform(9.5, 11.0))
    for i in blk:
        h0 = bv(i,"hac_energy")
        df_inj.at[i, "hac_energy"] = max(h0 * 1.55, h0 + 25.0)
        df_inj.at[i, "sim_supply_temp_c"] = sup_spike

def inject_7(blk):
    """图书馆高露点除湿"""
    for i in blk:
        h0 = bv(i,"hac_energy")
        df_inj.at[i, "sim_supply_temp_c"] = 5.5
        df_inj.at[i, "hac_energy"] = max(h0 * 2.0, h0 + 22.0)

def inject_8(blk):
    """图书馆闭馆忘关灯"""
    add_kw = float(rng_inj.uniform(50.0, 80.0))
    for i in blk:
        df_inj.at[i, "electricity_kwh"] = float(df_inj.at[i,"electricity_kwh"]) + add_kw

def inject_9(blk):
    """图书馆过渡季阀门内漏"""
    target_h = float(rng_inj.uniform(40.0, 60.0))
    for i in blk:
        df_inj.at[i, "hac_energy"] = target_h + float(rng_inj.normal(0, 1.5))
        sup = float(df_inj.at[i,"sim_supply_temp_c"])
        if sup > 10.0:
            df_inj.at[i, "sim_supply_temp_c"] = sup - rng_inj.uniform(0.3, 0.8)

def inject_10(blk):
    """图书馆供水传感器漂移"""
    spike = float(rng_inj.choice([3.5, 18.0]))
    for i in blk:
        df_inj.at[i, "sim_supply_temp_c"] = spike

def inject_11(blk):
    """办公楼纯非空调用电突增"""
    for i in blk:
        ee, he = bv(i,"electricity_kwh"), bv(i,"hac_energy")
        base_e = max(ee - he, 1.0)
        delta = max(30.0, 0.18 * ee, 0.11 * base_e + 8.0)
        df_inj.at[i, "electricity_kwh"] = float(df_inj.at[i,"electricity_kwh"]) + delta

def inject_12(blk):
    """办公楼夏季跑水"""
    mult = rng_inj.uniform(1.55, 1.95)
    for i in blk:
        df_inj.at[i, "water_l"] = float(df_inj.at[i,"water_l"]) * mult + 1100.0

def inject_13(blk):
    """办公楼回水温度传感器偏置"""
    bias = float(rng_inj.uniform(5.0, 8.0))
    for i in blk:
        df_inj.at[i, "sim_return_temp_c"] = bv(i,"sim_return_temp_c") + bias

INJECT_FN = {1:inject_1, 2:inject_2, 3:inject_3, 4:inject_4, 5:inject_5,
             6:inject_6, 7:inject_7, 8:inject_8, 9:inject_9, 10:inject_10,
             11:inject_11, 12:inject_12, 13:inject_13}

# ======== 5. 连续区间查找 ========
def find_block(month, bld_mask, dur_min, dur_max, extra=None):
    mask = (month_inj == month) & bld_mask & (~used)
    if extra is not None:
        cands = np.where(mask & extra)[0]
        if len(cands) == 0:
            cands = np.where(mask)[0]
    else:
        cands = np.where(mask)[0]
    if len(cands) == 0:
        return []
    dur = int(rng_inj.integers(dur_min, dur_max + 1))
    for _ in range(80):
        start = int(rng_inj.choice(cands))
        bid0 = df_inj.at[start, "building_id"]
        blk = []
        for k in range(dur + 10):
            j = start + k
            if j >= len(df_inj) or df_inj.at[j, "building_id"] != bid0: break
            if month_inj[j] != month or used[j]: break
            blk.append(j)
            if len(blk) >= dur: break
        if len(blk) >= dur_min:
            return blk[:max(dur_min, min(dur, len(blk)))]
    return []

# ======== 6. 异常配置 ========
CFG = {
    1:  {"name":"office_weekend_schedule_failure",      "bld":is_office,   "dur":(6,12),  "extra":dow_inj==5},
    2:  {"name":"office_masking_base_down_hvac_up",     "bld":is_office,   "dur":(3,5),   "extra":(dow_inj<5)&(hour_inj>=16)&(hour_inj<=19)},
    3:  {"name":"office_low_delta_t_bypass",            "bld":is_office,   "dur":(4,8),   "extra":None},
    4:  {"name":"teaching_water_leak",                  "bld":is_teaching, "dur":(4,8),   "extra":None},
    5:  {"name":"teaching_open_window_cooling_waste",   "bld":is_teaching, "dur":(3,6),   "extra":None},
    6:  {"name":"teaching_chiller_degradation_heatwave","bld":is_teaching, "dur":(3,5),   "extra":(hour_inj>=12)&(hour_inj<=16)},
    7:  {"name":"library_high_dew_dehumidification",    "bld":is_library,  "dur":(6,10),  "extra":None},
    8:  {"name":"library_night_lights_left_on",         "bld":is_library,  "dur":(4,7),   "extra":hour_inj<7},
    9:  {"name":"library_transition_valve_leak",        "bld":is_library,  "dur":(12,24), "extra":None},
    10: {"name":"library_supply_sensor_drift",          "bld":is_library,  "dur":(3,6),   "extra":None},
    11: {"name":"office_non_hvac_electricity_surge",    "bld":is_office,   "dur":(3,6),   "extra":(dow_inj<5)&(hour_inj>=10)&(hour_inj<=16)},
    12: {"name":"office_summer_water_leak",             "bld":is_office,   "dur":(4,8),   "extra":None},
    13: {"name":"office_return_temp_sensor_bias",       "bld":is_office,   "dur":(3,5),   "extra":None},
}

# ======== 7. 主循环：3~12 月 × 13 种异常 ========
for m in range(3, 13):
    for tid, c in CFG.items():
        blk = find_block(m, c["bld"], c["dur"][0], c["dur"][1], c["extra"])
        if not blk:
            print(f"[WARN] 月{m:2d} 类型{tid:2d} ({c['name']}): 无可用区间")
            continue
        INJECT_FN[tid](blk)
        for i in blk:
            used[i] = True
            labels.append({
                "timestamp": df_inj.at[i, "timestamp"],
                "building_id": df_inj.at[i, "building_id"],
                "anomaly_type_id": tid,
                "anomaly_name": c["name"],
                "anomaly_phase": "full",
            })

# ======== 8. 保存 ========
labels_df = pd.DataFrame(labels)

drop_cols = [c for c in df_inj.columns if c.endswith("_base")] + ["dayofweek"]
df_out = df_inj.drop(columns=[c for c in drop_cols if c in df_inj.columns])
df_out["timestamp"] = df_out["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_out.to_csv("data_2026_with_anomalies.csv", index=False, encoding="utf-8-sig")
labels_df.to_csv("data_2026_anomaly_labels.csv", index=False, encoding="utf-8-sig")

# ======== 9. 汇总 ========
print(f"注入完成: {len(labels_df)} 条标签, {len(df_out)} 行数据")
print(f"输出文件: data_2026_with_anomalies.csv / data_2026_baseline.csv / data_2026_anomaly_labels.csv")
print(f"\n各类型标签数:")
print(labels_df.groupby(["anomaly_type_id","anomaly_name"]).size()
      .reset_index(name="count").to_string(index=False))

labels_df["_month"] = pd.to_datetime(labels_df["timestamp"]).dt.month
pivot = labels_df.pivot_table(
    index="_month", columns="anomaly_type_id",
    values="anomaly_name", aggfunc="count", fill_value=0
)
print(f"\n月 × 类型矩阵（每格≥1 即满足覆盖要求）:\n{pivot}")
coverage = (pivot > 0).all(axis=1)
print(f"\n各月是否覆盖全部 13 种异常: {coverage.all()}  "
      f"({'全部覆盖' if coverage.all() else '缺失月份: ' + str(coverage[~coverage].index.tolist())})")

注入完成: 802 条标签, 26244 行数据
输出文件: data_2026_with_anomalies.csv / data_2026_baseline.csv / data_2026_anomaly_labels.csv

各类型标签数:
 anomaly_type_id                          anomaly_name  count
               1       office_weekend_schedule_failure     74
               2      office_masking_base_down_hvac_up     43
               3             office_low_delta_t_bypass     59
               4                   teaching_water_leak     63
               5    teaching_open_window_cooling_waste     43
               6 teaching_chiller_degradation_heatwave     38
               7     library_high_dew_dehumidification     74
               8          library_night_lights_left_on     54
               9         library_transition_valve_leak    170
              10           library_supply_sensor_drift     47
              11     office_non_hvac_electricity_surge     44
              12              office_summer_water_leak     56
              13        office_return_temp_sensor_bias     37

月 × 类型

In [15]:
import importlib, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "building_anomaly_detection.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import building_anomaly_detection as _det
importlib.reload(_det)
from building_anomaly_detection import (
    merge_expected_from_baseline,
    detect_building_kpi_anomalies,
    evaluate_symptom_recall,
)

# ======== 1. 读取注入后数据 + 基线 ========
df_det = pd.read_csv("data_2026_with_anomalies.csv")
df_det["timestamp"] = pd.to_datetime(df_det["timestamp"])
df_det = df_det.sort_values(["building_id","timestamp"]).reset_index(drop=True)

base_det = pd.read_csv("data_2026_baseline.csv")
base_det["timestamp"] = pd.to_datetime(base_det["timestamp"])

lab = pd.read_csv("data_2026_anomaly_labels.csv")
lab["timestamp"] = pd.to_datetime(lab["timestamp"])

# ======== 2. 合并期望列 & 检测 ========
df_det = merge_expected_from_baseline(df_det, base_det)
df_det = detect_building_kpi_anomalies(df_det, include_soft_symptoms=True)

# ======== 3. 总体症状统计 ========
print("=" * 60)
print("异常检测结果")
print("=" * 60)
for f in ["flag_water","flag_electricity","flag_hvac_energy",
          "flag_supply_deviation","flag_return_deviation"]:
    print(f"  {f:30s}: {int(df_det[f].sum()):>5d}")
print(f"  {'flag_symptom_any':30s}: {int(df_det['flag_symptom_any'].sum()):>5d}")

# ======== 4. 召回率评估 ========
res = evaluate_symptom_recall(df_det, lab, detection_mode="union")
print(f"\n标签小时数: {res['n_label_hours']}")
print(f"检测召回率 (union): {res['symptom_recall']:.4f}")

if "recall_by_anomaly_type" in res:
    print("\n各异常类型召回率:")
    print(res["recall_by_anomaly_type"].to_string())

# ======== 5. 未检出行分析 ========
missed = res.get("missed_keys", pd.DataFrame())
if len(missed) > 0:
    missed_with_lab = missed.merge(lab, on=["timestamp","building_id"], how="left")
    print(f"\n未检出标签行: {len(missed)} 条")
    missed_types = missed_with_lab.groupby("anomaly_type_id").size()
    print("未检出按类型分布:")
    print(missed_types)
else:
    print("\n所有标签行均被检出，召回率 100%。")

异常检测结果
  flag_water                    :   119
  flag_electricity              :   135
  flag_hvac_energy              :   465
  flag_supply_deviation         :   170
  flag_return_deviation         :   154
  flag_symptom_any              :   770

标签小时数: 802
检测召回率 (union): 0.9888

各异常类型召回率:
anomaly_type_id
1     1.000000
2     1.000000
3     1.000000
4     1.000000
5     1.000000
6     1.000000
7     1.000000
8     1.000000
9     0.947059
10    1.000000
11    1.000000
12    1.000000
13    1.000000

未检出标签行: 9 条
未检出按类型分布:
anomaly_type_id
9    9
dtype: int64


In [21]:
df_rename = pd.read_csv("data_2026_with_anomalies.csv")
df_rename = df_rename.rename(columns={
    "airTemperature": "air_temp_c",
    "dewTemperature": "dew_temp_c",
    "water_l": "water_m3",
})
df_rename = df_rename.drop(columns=["primaryspaceusage", "sqm"], errors="ignore")
df_rename.to_csv("data_2026_with_anomalies.csv", index=False, encoding="utf-8-sig")

print("列名已重命名:")
print("  airTemperature → air_temp_c")
print("  dewTemperature → dew_temp_c")
print("  water_l        → water_m3")
print(f"\n当前列: {list(df_rename.columns)}")

列名已重命名:
  airTemperature → air_temp_c
  dewTemperature → dew_temp_c
  water_l        → water_m3

当前列: ['timestamp', 'building_id', 'air_temp_c', 'dew_temp_c', 'humidity_rh_pct', 'water_m3', 'electricity_kwh', 'sim_supply_temp_c', 'sim_return_temp_c', 'hac_energy']


In [22]:
df_split = pd.read_csv("data_2026_with_anomalies.csv")
df_split["timestamp"] = pd.to_datetime(df_split["timestamp"])

first3 = df_split[df_split["timestamp"].dt.month <= 3]
last9  = df_split[df_split["timestamp"].dt.month >= 4]

first3.to_csv("data_2026_jan_mar.csv", index=False, encoding="utf-8-sig")
last9.to_csv("data_2026_apr_dec.csv", index=False, encoding="utf-8-sig")

print(f"前三个月 (1-3月): {len(first3)} 行 → data_2026_jan_mar.csv")
print(f"后九个月 (4-12月): {len(last9)} 行 → data_2026_apr_dec.csv")
print(f"合计: {len(first3) + len(last9)} 行")

前三个月 (1-3月): 6453 行 → data_2026_jan_mar.csv
后九个月 (4-12月): 19791 行 → data_2026_apr_dec.csv
合计: 26244 行


In [ ]:
import importlib, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "building_anomaly_detection.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import building_anomaly_detection as _det
importlib.reload(_det)
from building_anomaly_detection import merge_expected_from_baseline, detect_building_kpi_anomalies

# ---- 读取前三个月数据（需要还原列名以匹配基线）----
df_q1 = pd.read_csv("data_2026_jan_mar.csv")
df_q1["timestamp"] = pd.to_datetime(df_q1["timestamp"])
df_q1 = df_q1.rename(columns={
    "air_temp_c": "airTemperature",
    "dew_temp_c": "dewTemperature",
    "water_m3": "water_l",
})
df_q1 = df_q1.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

# ---- 读取基线（列名未改，与检测模块一致）----
base_q1 = pd.read_csv("data_2026_baseline.csv")
base_q1["timestamp"] = pd.to_datetime(base_q1["timestamp"])

# ---- 合并 & 检测 ----
df_q1 = merge_expected_from_baseline(df_q1, base_q1)
df_q1 = detect_building_kpi_anomalies(df_q1, include_soft_symptoms=True)

# ---- 汇总 ----
print("=" * 60)
print("前三个月 (1-3月) 异常检测结果")
print("=" * 60)
for f in ["flag_water","flag_electricity","flag_hvac_energy",
          "flag_supply_deviation","flag_return_deviation"]:
    print(f"  {f:30s}: {int(df_q1[f].sum()):>5d}")
print(f"  {'flag_symptom_any':30s}: {int(df_q1['flag_symptom_any'].sum()):>5d}")
print(f"  {'flag_symptom_soft_any':30s}: {int(df_q1['flag_symptom_soft_any'].sum()):>5d}")
print(f"  {'flag_symptom_union_any':30s}: {int(df_q1['flag_symptom_union_any'].sum()):>5d}")

n_total = len(df_q1)
n_any = int(df_q1["flag_symptom_union_any"].sum())
print(f"\n总行数: {n_total}，被标记行数: {n_any}，异常率: {n_any/n_total:.4%}")

if n_any > 0:
    show_cols = ["timestamp","building_id",
                 "flag_water","flag_electricity","flag_hvac_energy",
                 "flag_supply_deviation","flag_return_deviation",
                 "hvac_diagnosis"]
    print("\n被标记的样例行:")
    display(df_q1.loc[df_q1["flag_symptom_union_any"], show_cols].head(10))
else:
    print("\n前三个月无任何异常标记（1-2月无注入，3月注入的异常均可检出）。")